In [ ]:
import pandas as pd
import pandas as pd

In [ ]:
df = pd.read_excel('/content/pos_tagged.xlsx')

# Accessing the first column as it appears to be the one containing sentences
print("First 5 sentences:")
for i, sentence in enumerate(df.iloc[:, 0].head(5)):
    print(f"{i+1}. {sentence}")

First 5 sentences:
1. 1\QT_QTC .\RD_PUNC 0\QT_QTC .\RD_PUNC
2. थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP असम\N_NNP कमेनस्कि\N_NNP (\RD_PUNC Jan\N_NNP Amos\N_NNP Komensky\N_NNP )\RD_PUNC आ\PSP फोसावनाय\V_VM सावगारि\N_NN बिजाब\N_NN Orbis\N_NNP Pictus\N_NNP खौनो\PSP गथ’\N_NNP थुनलाइनि\N_NNP गिबि\JJ बिजाब\N_NN होन्ना\V_VM साननाय\V_VM जायो\V_VM ।\RD_PUNC
3. मुलुगनि\N_NN गुबुन\JJ गुबुन\V_VAUX थुनलाइनि\N_NNP बादिनो\PSP बर’\N_NNP थुनलायावबो\N_NNP गथ’\N_NNP थुनलाइनि\N_NNP थि\JJ जोनोमखौ\N_NN बुंनो\V_VM गोब्राब\RB जायो\V_VM ।\RD_PUNC
4. खुगाजों\N_NN खुगा\N_NN सोलिबोनाय\V_VM गथ’\N_NNP थुनलाया\N_NNP सिगांनिफ्रायनो\N_NST सोलिबोसेयावबो\V_VM बेनो\DM_DMD लिरनाय\V_VM महर\PSP होजेन्दोंमोन\V_VM मिसनारिफोरा\N_NNP 19\N_NN जौथाइनि\N_NN जोबनायथिं\V_VM ।\RD_PUNC
5. मिसनारिफोरा\N_NN रनसायनाय\V_VM आरो\CC_CCD रावसोलायनाय\V_VM बाइबेलनि\N_NNP सल’\N_NN ,\RD_PUNC गोजाम\JJ रादाइनि\N_NN सल’\N_NN ,\RD_PUNC जीशुनि\N_NNP जिउखौरां\N_NN ,\RD_PUNC जीशुनि\N_NNP मावनाय\V_VM दांनायफोरानो\V_VM बेनि\DM_DMD बिदिन्थि\N_NN ।\RD_

In [ ]:
import re
import pandas as pd

def parse_sentence(sentence_string):
    if not isinstance(sentence_string, str):
        return [], []
    words = []
    tags = []
    # Split by one or more whitespace characters and filter out empty strings
    pairs = [p for p in re.split(r'\s+', sentence_string) if p]
    for pair in pairs:
        parts = pair.rsplit('\\', 1)
        if len(parts) == 2:
            words.append(parts[0])
            tags.append(parts[1])
        else:
            # If no '\\' is found, treat the whole part as a word and tag as empty or 'UNK'
            words.append(parts[0])
            tags.append('UNK') # Or any other placeholder for unknown tags
    return words, tags

# Apply the function to the first column (df.iloc[:, 0]) of the DataFrame
df[['words', 'tags']] = df.iloc[:, 0].apply(lambda x: pd.Series(parse_sentence(x)))

# Display the DataFrame with the new columns
print("DataFrame with 'words' and 'tags' columns after initial parsing:")
display(df.head())

DataFrame with 'words' and 'tags' columns after initial parsing:


,Value,words,tags
0,1\QT_QTC .\RD_PUNC 0\QT_QTC .\RD_PUNC,"[1, ., 0, .]","[QT_QTC, RD_PUNC, QT_QTC, RD_PUNC]"
1,थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP ...,"[थेवबो, 1658, मायथायाव, जन, असम, कमेनस्कि, (, ...","[CC_CCS, N_NN, N_NN, N_NNP, N_NNP, N_NNP, RD_P..."
2,मुलुगनि\N_NN गुबुन\JJ गुबुन\V_VAUX थुनलाइनि\N_...,"[मुलुगनि, गुबुन, गुबुन, थुनलाइनि, बादिनो, बर’,...","[N_NN, JJ, V_VAUX, N_NNP, PSP, N_NNP, N_NNP, N..."
3,खुगाजों\N_NN खुगा\N_NN सोलिबोनाय\V_VM गथ’\N_NN...,"[खुगाजों, खुगा, सोलिबोनाय, गथ’, थुनलाया, सिगां...","[N_NN, N_NN, V_VM, N_NNP, N_NNP, N_NST, V_VM, ..."
4,मिसनारिफोरा\N_NN रनसायनाय\V_VM आरो\CC_CCD रावस...,"[मिसनारिफोरा, रनसायनाय, आरो, रावसोलायनाय, बाइब...","[N_NN, V_VM, CC_CCD, V_VM, N_NNP, N_NN, RD_PUN..."


In [ ]:
def remove_duplicate_word_tags(words, tags):
    seen = set()
    unique_words = []
    unique_tags = []
    for word, tag in zip(words, tags):
        word_tag_pair = (word, tag) # Use tuple for hashability
        if word_tag_pair not in seen:
            seen.add(word_tag_pair)
            unique_words.append(word)
            unique_tags.append(tag)
    return unique_words, unique_tags

# Apply the function to remove duplicate word-tag pairs within each sentence
df[['words', 'tags']] = df.apply(lambda row: pd.Series(remove_duplicate_word_tags(row['words'], row['tags'])), axis=1)

# Remove duplicate sentences (based on the original 'Value' column or 'words'/'tags' if you prefer a 'cleaned' sentence uniqueness)
# For this, let's join words back to form a 'cleaned_sentence' string to detect duplicates reliably
df['cleaned_sentence'] = df['words'].apply(lambda x: ' '.join(x))
df.drop_duplicates(subset=['cleaned_sentence'], inplace=True)

# Drop the 'cleaned_sentence' helper column if not needed further
df.drop(columns=['cleaned_sentence'], inplace=True)

print("DataFrame after removing duplicate word-tag pairs per sentence and duplicate sentences:")
display(df.head())
print(f"New DataFrame shape: {df.shape}")

DataFrame after removing duplicate word-tag pairs per sentence and duplicate sentences:


,Value,words,tags
0,1\QT_QTC .\RD_PUNC 0\QT_QTC .\RD_PUNC,"[1, ., 0]","[QT_QTC, RD_PUNC, QT_QTC]"
1,थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP ...,"[थेवबो, 1658, मायथायाव, जन, असम, कमेनस्कि, (, ...","[CC_CCS, N_NN, N_NN, N_NNP, N_NNP, N_NNP, RD_P..."
2,मुलुगनि\N_NN गुबुन\JJ गुबुन\V_VAUX थुनलाइनि\N_...,"[मुलुगनि, गुबुन, गुबुन, थुनलाइनि, बादिनो, बर’,...","[N_NN, JJ, V_VAUX, N_NNP, PSP, N_NNP, N_NNP, N..."
3,खुगाजों\N_NN खुगा\N_NN सोलिबोनाय\V_VM गथ’\N_NN...,"[खुगाजों, खुगा, सोलिबोनाय, गथ’, थुनलाया, सिगां...","[N_NN, N_NN, V_VM, N_NNP, N_NNP, N_NST, V_VM, ..."
4,मिसनारिफोरा\N_NN रनसायनाय\V_VM आरो\CC_CCD रावस...,"[मिसनारिफोरा, रनसायनाय, आरो, रावसोलायनाय, बाइब...","[N_NN, V_VM, CC_CCD, V_VM, N_NNP, N_NN, RD_PUN..."


New DataFrame shape: (5989, 3)


In [ ]:
all_tags = [tag for sublist in df['tags'] for tag in sublist]
tag_distribution = pd.Series(all_tags).value_counts()

print("Distribution of each tag in the dataset:")
display(tag_distribution)

Distribution of each tag in the dataset:


,count
N_NN,22367
V_VM,16949
RD_PUNC,10400
N_NNP,9420
JJ,4928
N_NST,3264
QT_QTC,2308
CC_CCD,2236
DM_DMD,2230
PSP,1856


In [ ]:
pip install transformers

In [ ]:
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = "ai4bharat/IndicBERT-v3-270M"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Define a function to tokenize a list of words
def tokenize_words(word_list):
    # Ensure word_list is not empty to avoid tokenizer errors
    if not word_list:
        return {'input_ids': [], 'attention_mask': [], 'word_ids': []}

    # Tokenize the list of words
    encoding = tokenizer(
        word_list,
        is_split_into_words=True,
        truncation=True,
        padding=False # Do not pad here; we want dynamic lengths
    )
    # Store word_ids as a list, and convert input_ids and attention_mask to lists as well
    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'word_ids': encoding.word_ids()
    }

# Apply the tokenization function to the 'words' column
df['tokenized_data'] = df['words'].apply(tokenize_words)

# Extract input_ids, attention_mask, and word_ids into separate columns for clarity
df['input_ids'] = df['tokenized_data'].apply(lambda x: x['input_ids'])
df['attention_mask'] = df['tokenized_data'].apply(lambda x: x['attention_mask'])
df['word_ids'] = df['tokenized_data'].apply(lambda x: x['word_ids'])

# Drop the intermediate 'tokenized_data' column if not needed
df.drop(columns=['tokenized_data'], inplace=True)

print("DataFrame with tokenized input_ids, attention_mask, and word_ids:")
display(df.head())

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/802 [00:00<?, ?B/s]

DataFrame with tokenized input_ids, attention_mask, and word_ids:


,Value,words,tags,input_ids,attention_mask,word_ids
0,1\QT_QTC .\RD_PUNC 0\QT_QTC .\RD_PUNC,"[1, ., 0]","[QT_QTC, RD_PUNC, QT_QTC]","[2, 236770, 236761, 236771]","[1, 1, 1, 1]","[None, 0, 1, 2]"
1,थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP ...,"[थेवबो, 1658, मायथायाव, जन, असम, कमेनस्कि, (, ...","[CC_CCS, N_NN, N_NN, N_NNP, N_NNP, N_NNP, RD_P...","[2, 30935, 236869, 38540, 236770, 236825, 2368...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, ..."
2,मुलुगनि\N_NN गुबुन\JJ गुबुन\V_VAUX थुनलाइनि\N_...,"[मुलुगनि, गुबुन, गुबुन, थुनलाइनि, बादिनो, बर’,...","[N_NN, JJ, V_VAUX, N_NNP, PSP, N_NNP, N_NNP, N...","[2, 236845, 5735, 236903, 236886, 10047, 18415...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, ..."
3,खुगाजों\N_NN खुगा\N_NN सोलिबोनाय\V_VM गथ’\N_NN...,"[खुगाजों, खुगा, सोलिबोनाय, गथ’, थुनलाया, सिगां...","[N_NN, N_NN, V_VM, N_NNP, N_NNP, N_NST, V_VM, ...","[2, 82364, 2409, 20011, 82364, 2409, 148976, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, ..."
4,मिसनारिफोरा\N_NN रनसायनाय\V_VM आरो\CC_CCD रावस...,"[मिसनारिफोरा, रनसायनाय, आरो, रावसोलायनाय, बाइब...","[N_NN, V_VM, CC_CCD, V_VM, N_NNP, N_NN, RD_PUN...","[2, 236845, 3052, 236833, 867, 21214, 163634, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 3, ..."


In [ ]:
# Display an example of the tokenization for the first sentence
first_sentence_words = df['words'].iloc[0]
first_sentence_input_ids = df['input_ids'].iloc[0]
first_sentence_word_ids = df['word_ids'].iloc[0]

print("Original Words (first sentence):")
print(first_sentence_words)

print("\nTokens (first sentence):")
print(tokenizer.convert_ids_to_tokens(first_sentence_input_ids))

print("\nWord IDs (first sentence):")
print(first_sentence_word_ids)

Original Words (first sentence):
['1', '.', '0']

Tokens (first sentence):
['<bos>', '1', '.', '0']

Word IDs (first sentence):
[None, 0, 1, 2]


In [ ]:
# 1. Create a tag-to-ID mapping

# Get all unique tags from the 'tags' column
all_unique_tags = sorted(list(set(tag for sublist in df['tags'] for tag in sublist)))

# Add a special tag for tokens that are not the first part of a word, e.g., for padding or subword continuations
# -100 is a common choice that many Hugging Face models ignore in loss calculation
tag_to_id = {tag: i for i, tag in enumerate(all_unique_tags)}
tag_to_id['[PAD]'] = -100 # Special ID for padding tokens

id_to_tag = {i: tag for tag, i in tag_to_id.items() if i != -100}

print("Tag to ID mapping:")
display(tag_to_id)
print("\nID to Tag mapping:")
display(id_to_tag)

Tag to ID mapping:


{'CC_CCD': 0,
 'CC_CCS': 1,
 'DM_DMD': 2,
 'DM_DMI': 3,
 'DM_DMQ': 4,
 'DM_DMR': 5,
 'JJ': 6,
 'N_NN': 7,
 'N_NNP': 8,
 'N_NST': 9,
 'PR_PRC': 10,
 'PR_PRF': 11,
 'PR_PRI': 12,
 'PR_PRL': 13,
 'PR_PRP': 14,
 'PR_PRQ': 15,
 'PSP': 16,
 'QT_QTC': 17,
 'QT_QTF': 18,
 'QT_QTO': 19,
 'RB': 20,
 'RD_ECH': 21,
 'RD_PUNC': 22,
 'RD_RDF': 23,
 'RD_SYM': 24,
 'RD_UNK': 25,
 'RP_INJ': 26,
 'RP_INTF': 27,
 'RP_NEG': 28,
 'RP_RPD': 29,
 'V_VAUX': 30,
 'V_VAUX_VF': 31,
 'V_VM': 32,
 'V_VM_VF': 33,
 'V_VM_VNF': 34,
 '[PAD]': -100}


ID to Tag mapping:


{0: 'CC_CCD',
 1: 'CC_CCS',
 2: 'DM_DMD',
 3: 'DM_DMI',
 4: 'DM_DMQ',
 5: 'DM_DMR',
 6: 'JJ',
 7: 'N_NN',
 8: 'N_NNP',
 9: 'N_NST',
 10: 'PR_PRC',
 11: 'PR_PRF',
 12: 'PR_PRI',
 13: 'PR_PRL',
 14: 'PR_PRP',
 15: 'PR_PRQ',
 16: 'PSP',
 17: 'QT_QTC',
 18: 'QT_QTF',
 19: 'QT_QTO',
 20: 'RB',
 21: 'RD_ECH',
 22: 'RD_PUNC',
 23: 'RD_RDF',
 24: 'RD_SYM',
 25: 'RD_UNK',
 26: 'RP_INJ',
 27: 'RP_INTF',
 28: 'RP_NEG',
 29: 'RP_RPD',
 30: 'V_VAUX',
 31: 'V_VAUX_VF',
 32: 'V_VM',
 33: 'V_VM_VF',
 34: 'V_VM_VNF'}

In [ ]:
# 2. Align tags with tokens and convert them to IDs

def align_labels_with_tokens(tags, word_ids):
    aligned_labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None: # Special tokens like [CLS], [SEP], [PAD]
            aligned_labels.append(-100) # Ignore these in loss calculation
        elif word_idx != previous_word_idx: # Start of a new word
            # Map the original word's tag to its corresponding token's ID
            if word_idx < len(tags):
                aligned_labels.append(tag_to_id[tags[word_idx]])
            else:
                # Handle cases where word_idx might be out of bounds for tags list
                # This could happen if truncation cut off words from original text but word_ids still refer to them
                aligned_labels.append(-100) # Or a specific 'UNK' tag ID if available
        else: # Continuation of a word that was split into subword tokens
            aligned_labels.append(-100) # Ignore subword continuations for sequence labeling task

        previous_word_idx = word_idx
    return aligned_labels

# Apply the alignment function to create a new 'labels' column
df['labels'] = df.apply(lambda row: align_labels_with_tokens(row['tags'], row['word_ids']), axis=1)

print("DataFrame with aligned numerical labels:")
display(df.head())

DataFrame with aligned numerical labels:


,Value,words,tags,input_ids,attention_mask,word_ids,labels
0,1\QT_QTC .\RD_PUNC 0\QT_QTC .\RD_PUNC,"[1, ., 0]","[QT_QTC, RD_PUNC, QT_QTC]","[2, 236770, 236761, 236771]","[1, 1, 1, 1]","[None, 0, 1, 2]","[-100, 17, 22, 17]"
1,थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP ...,"[थेवबो, 1658, मायथायाव, जन, असम, कमेनस्कि, (, ...","[CC_CCS, N_NN, N_NN, N_NNP, N_NNP, N_NNP, RD_P...","[2, 30935, 236869, 38540, 236770, 236825, 2368...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, ...","[-100, 1, -100, -100, 7, -100, -100, -100, 7, ..."
2,मुलुगनि\N_NN गुबुन\JJ गुबुन\V_VAUX थुनलाइनि\N_...,"[मुलुगनि, गुबुन, गुबुन, थुनलाइनि, बादिनो, बर’,...","[N_NN, JJ, V_VAUX, N_NNP, PSP, N_NNP, N_NNP, N...","[2, 236845, 5735, 236903, 236886, 10047, 18415...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, ...","[-100, 7, -100, -100, -100, -100, 6, -100, -10..."
3,खुगाजों\N_NN खुगा\N_NN सोलिबोनाय\V_VM गथ’\N_NN...,"[खुगाजों, खुगा, सोलिबोनाय, गथ’, थुनलाया, सिगां...","[N_NN, N_NN, V_VM, N_NNP, N_NNP, N_NST, V_VM, ...","[2, 82364, 2409, 20011, 82364, 2409, 148976, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, ...","[-100, 7, -100, -100, 7, -100, 32, -100, -100,..."
4,मिसनारिफोरा\N_NN रनसायनाय\V_VM आरो\CC_CCD रावस...,"[मिसनारिफोरा, रनसायनाय, आरो, रावसोलायनाय, बाइब...","[N_NN, V_VM, CC_CCD, V_VM, N_NNP, N_NN, RD_PUN...","[2, 236845, 3052, 236833, 867, 21214, 163634, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 3, ...","[-100, 7, -100, -100, -100, -100, -100, 32, -1..."


In [ ]:
# Display an example of the aligned labels for the first sentence
first_sentence_words = df['words'].iloc[0]
first_sentence_input_ids = df['input_ids'].iloc[0]
first_sentence_word_ids = df['word_ids'].iloc[0]
first_sentence_tags = df['tags'].iloc[0]
first_sentence_labels = df['labels'].iloc[0]

print("Original Words (first sentence):")
print(first_sentence_words)

print("Original Tags (first sentence):")
print(first_sentence_tags)

print("\nTokens (first sentence):")
print(tokenizer.convert_ids_to_tokens(first_sentence_input_ids))

print("\nWord IDs (first sentence):")
print(first_sentence_word_ids)

print("\nAligned Labels (first sentence):")
print(first_sentence_labels)

print("\nAligned Labels mapped back to Tags (first sentence):")
# Filter out -100 for display clarity
print([id_to_tag[label_id] for label_id in first_sentence_labels if label_id != -100])

Original Words (first sentence):
['1', '.', '0']
Original Tags (first sentence):
['QT_QTC', 'RD_PUNC', 'QT_QTC']

Tokens (first sentence):
['<bos>', '1', '.', '0']

Word IDs (first sentence):
[None, 0, 1, 2]

Aligned Labels (first sentence):
[-100, 17, 22, 17]

Aligned Labels mapped back to Tags (first sentence):
['QT_QTC', 'RD_PUNC', 'QT_QTC']


In [ ]:
from sklearn.model_selection import train_test_split

# Split the DataFrame into training, validation, and test sets
# First, split into train (80%) and temp (20%)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

# Then, split temp (20%) into validation (10%) and test (10%)
# Since temp_df is 20% of the original, test_size=0.5 will make val_df and test_df 10% each
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

# Display the head of the training DataFrame as an example
print("\nTraining DataFrame head:")
display(train_df.head())

Training set shape: (4791, 7)
Validation set shape: (599, 7)
Test set shape: (599, 7)

Training DataFrame head:


,Value,words,tags,input_ids,attention_mask,word_ids,labels
3814,खैनानि\N_NN लोगोआव\N_NN बैराथि\N_NN आरो\CC_CCD...,"[खैनानि, लोगोआव, बैराथि, आरो, लोगो, सिख्लाफोरा...","[N_NN, N_NN, N_NN, CC_CCD, N_NN, N_NN, V_VM, R...","[2, 236975, 236883, 1238, 10047, 95232, 236850...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, ...","[-100, 7, -100, -100, -100, 7, -100, -100, 7, ..."
5399,चिरांनि\N_NNP राजेश\N_NNP थापाया\N_NNP बेष्ट\R...,"[चिरांनि, राजेश, थापाया, बेष्ट, अब, दा, बेष्टस...","[N_NNP, N_NNP, N_NNP, RD_UNK, RD_UNK, RD_UNK, ...","[2, 47695, 130096, 10047, 18936, 2387, 10583, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 1, 1, 2, 2, 2, 3, 3, 4, 5, 6, ...","[-100, 8, -100, -100, 8, -100, 8, -100, -100, ..."
5971,आसाम\N_NNP ब’डी\N_NNP बिल्डारस\N_NNP एस’सियेछन...,"[आसाम, ब’डी, बिल्डारस, एस’सियेछननि, बानजायनाया...","[N_NNP, N_NNP, N_NNP, N_NNP, N_NN, V_VM, DM_DM...","[2, 236930, 98763, 236885, 236858, 7734, 23688...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, ...","[-100, 8, -100, 8, -100, -100, 8, -100, -100, ..."
1965,समायना\JJ रमायना\JJ मुं\N_NST लाखिनायाव\N_NST ...,"[समायना, रमायना, मुं, लाखिनायाव, बिसोरनि, हाइन...","[JJ, JJ, N_NST, N_NST, PR_PRP, JJ, N_NST, JJ, ...","[2, 103035, 236856, 1238, 236805, 2380, 236856...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[None, 0, 0, 0, 1, 1, 1, 1, 2, 2, 3, 3, 3, 3, ...","[-100, 6, -100, -100, 6, -100, -100, -100, 9, ..."
829,बर’\N_NNP समाजा\N_NN बिफा\N_NN गाहाय\JJ समाज\N...,"[बर’, समाजा, बिफा, गाहाय, समाज, ।]","[N_NNP, N_NN, N_NN, JJ, N_NN, RD_PUNC]","[2, 4365, 236858, 103035, 5779, 236885, 21214,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[None, 0, 0, 1, 1, 2, 2, 2, 3, 3, 4, 5]","[-100, 8, -100, 7, -100, 7, -100, -100, 6, -10..."


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import DataCollatorForTokenClassification

# Define a custom PyTorch Dataset class
class POSDataset(Dataset):
    def __init__(self, dataframe):
        self.input_ids = [torch.tensor(ids, dtype=torch.long) for ids in dataframe['input_ids'].tolist()]
        self.attention_mask = [torch.tensor(mask, dtype=torch.long) for mask in dataframe['attention_mask'].tolist()]
        self.labels = [torch.tensor(lbls, dtype=torch.long) for lbls in dataframe['labels'].tolist()]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create instances of the custom Dataset for each split
train_dataset = POSDataset(train_df)
val_dataset = POSDataset(val_df)
test_dataset = POSDataset(test_df)

print(f"Number of samples in training dataset: {len(train_dataset)}")
print(f"Number of samples in validation dataset: {len(val_dataset)}")
print(f"Number of samples in test dataset: {len(test_dataset)}")

# Initialize the DataCollator for token classification
# The tokenizer is needed by the data collator for padding tokens
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)

print("\nDataCollatorForTokenClassification initialized. Ready for batching!")

# Display a sample from the training dataset to show its structure
print("\nFirst sample from training dataset:")
sample = train_dataset[0]
for key, value in sample.items():
    print(f"{key}: {value.shape} - {value[:5].tolist()}...")

Number of samples in training dataset: 4791
Number of samples in validation dataset: 599
Number of samples in test dataset: 599

DataCollatorForTokenClassification initialized. Ready for batching!

First sample from training dataset:
input_ids: torch.Size([26]) - [2, 236975, 236883, 1238, 10047]...
attention_mask: torch.Size([26]) - [1, 1, 1, 1, 1]...
labels: torch.Size([26]) - [-100, 7, -100, -100, -100]...


In [ ]:
print("Format of a single entry in the dataset:")
sample_entry = train_dataset[0]
for key, value in sample_entry.items():
    print(f"Key: {key}, Type: {type(value)}, Shape: {value.shape}, Example values (first 5): {value[:5].tolist()}")

Format of a single entry in the dataset:
Key: input_ids, Type: <class 'torch.Tensor'>, Shape: torch.Size([26]), Example values (first 5): [2, 236975, 236883, 1238, 10047]
Key: attention_mask, Type: <class 'torch.Tensor'>, Shape: torch.Size([26]), Example values (first 5): [1, 1, 1, 1, 1]
Key: labels, Type: <class 'torch.Tensor'>, Shape: torch.Size([26]), Example values (first 5): [-100, 7, -100, -100, -100]


In [ ]:
!pip install evaluate
import numpy as np
import evaluate

from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [ ]:
MODEL_NAME = "ai4bharat/IndicBERT-v3-270M"

label2id = tag_to_id
id2label = id_to_tag

In [ ]:
print(MODEL_NAME)

ai4bharat/IndicBERT-v3-270M


In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print(type(config))

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


<class 'transformers.models.gemma3.configuration_gemma3.Gemma3TextConfig'>


checking the dataset

In [ ]:
print(train_dataset[0])

{'input_ids': tensor([     2, 236975, 236883,   1238,  10047,  95232, 236850,  81762,  74315,
          2208,  29494, 236930,   5409,  95232, 236850, 236831,   5913,  30528,
         77115,   2208, 236878, 239558,   1238,   1981, 237172, 236890]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1]), 'labels': tensor([-100,    7, -100, -100, -100,    7, -100, -100,    7, -100, -100,    0,
        -100,    7, -100,    7, -100, -100, -100, -100,   32, -100, -100, -100,
        -100,   22])}


In [ ]:
from transformers import AutoModelForCausalLM

MODEL_NAME = "ai4bharat/IndicBERT-v3-270M"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    output_hidden_states=True
)

print(base_model)

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


bidirection_gemma3.py:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ai4bharat/IndicBERT-v3-270M:
- bidirection_gemma3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

BidirectionalGemma3ForCausalLM(
  (model): BidirectionalGemma3Model(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x NonCausalGemma3DecoderLayer(
        (self_attn): NonCausalGemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_laye

In [ ]:
print(base_model.config)
print("Hidden Size:", base_model.config.hidden_size)

Gemma3TextConfig {
  "_sliding_window_pattern": 6,
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "auto_map": {
    "AutoModel": "bidirection_gemma3.BidirectionalGemma3ForCausalLM",
    "AutoModelForCausalLM": "bidirection_gemma3.BidirectionalGemma3ForCausalLM"
  },
  "bos_token_id": 2,
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": null,
  "head_dim": 256,
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 640,
  "initializer_range": 0.02,
  "intermediate_size": 2048,
  "is_causal": false,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
   

In [ ]:
print(base_model)


BidirectionalGemma3ForCausalLM(
  (model): BidirectionalGemma3Model(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x NonCausalGemma3DecoderLayer(
        (self_attn): NonCausalGemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_laye

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM
from transformers.modeling_outputs import TokenClassifierOutput


class IndicBERTv3ForPOSTagging(nn.Module):

    def __init__(self, model_name, num_labels):
        super().__init__()

        # Load IndicBERT v3 backbone
        self.backbone = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True
        )

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            self.backbone.config.hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
    ):

        # Forward pass through IndicBERT v3 backbone
        outputs = self.backbone.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

        # Last hidden state
        sequence_output = outputs.last_hidden_state

        # Convert from bfloat16 -> float32
        sequence_output = self.dropout(sequence_output.float())

        # Token classification
        logits = self.classifier(sequence_output)

        loss = None

        if labels is not None:

            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

            loss = loss_fct(
                logits.view(-1, logits.size(-1)),
                labels.view(-1)
            )

        return TokenClassifierOutput(
            loss=loss,
            logits=logits
        )

In [ ]:
model = IndicBERTv3ForPOSTagging(
    MODEL_NAME,
    num_labels=len(label2id)
)

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [ ]:
sample = train_dataset[0]

input_ids = sample["input_ids"].unsqueeze(0)
attention_mask = sample["attention_mask"].unsqueeze(0)
labels = sample["labels"].unsqueeze(0)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

print(outputs["loss"])
print(outputs["logits"].shape)

tensor(14.0092, grad_fn=<NllLossBackward0>)
torch.Size([1, 26, 36])


In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    "ai4bharat/IndicBERT-v3-270M",
    trust_remote_code=True
)

print(model)

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

BidirectionalGemma3ForCausalLM(
  (model): BidirectionalGemma3Model(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x NonCausalGemma3DecoderLayer(
        (self_attn): NonCausalGemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_laye

In [ ]:
from transformers.modeling_outputs import TokenClassifierOutput
import torch # Import torch for Parameter and clone
from transformers import AutoModelForCausalLM
import torch.nn as nn

class IndicBERTv3ForPOSTagging(nn.Module):

    def __init__(self, model_name, num_labels):
        super().__init__()

        # Load IndicBERT v3 backbone
        self.backbone = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True
        )

        # Explicitly untie the weights for the lm_head and embeddings
        # to prevent RuntimeError with safetensors during saving.
        # This is applicable if the model's embedding and output layers share weights.
        if hasattr(self.backbone, 'untie_weights'):
            self.backbone.untie_weights()
            print(f"Untied weights for {model_name} using model.untie_weights().")
        else:
            # Fallback for models without untie_weights method or older versions
            # Manually untie if lm_head and embed_tokens share weights
            if hasattr(self.backbone, 'lm_head') and hasattr(self.backbone.model, 'embed_tokens'):
                if self.backbone.lm_head.weight is self.backbone.model.embed_tokens.weight:
                    self.backbone.lm_head.weight = torch.nn.Parameter(self.backbone.model.embed_tokens.weight.clone().detach())
                    print("Manually untied lm_head and embed_tokens weights via cloning.")

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            self.backbone.config.hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
    ):

        # Forward pass through IndicBERT v3 backbone
        outputs = self.backbone.model( # Access the 'model' attribute within backbone for hidden states
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

        # Last hidden state
        sequence_output = outputs.last_hidden_state

        # Convert from bfloat16 -> float32 for dropout and linear layer if necessary
        # The .float() conversion might be important if the model is loaded in bfloat16 (as per config)
        sequence_output = self.dropout(sequence_output.float())

        # Token classification
        logits = self.classifier(sequence_output)

        loss = None

        if labels is not None:
            # Use CrossEntropyLoss for token classification
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100) # -100 labels are ignored

            # Reshape logits and labels for loss calculation
            loss = loss_fct(
                logits.view(-1, logits.size(-1)), # (batch_size * sequence_length, num_labels)
                labels.view(-1) # (batch_size * sequence_length)
            )

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states, # Keep hidden states for potential debugging/analysis
            attentions=outputs.attentions, # Keep attentions if available
        )

In [ ]:
import transformers
print(transformers.__version__)

5.13.1


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=5cee93aceabaaae7de5a8d57a1969fad318f5ff6534ab6b3676b9a92e90c32db
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

In [ ]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        pred_tags = []
        gold_tags = []

        for pred, lab in zip(prediction, label):

            if lab != -100:
                pred_tags.append(id2label[pred])
                gold_tags.append(id2label[lab])

        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    return {
        "accuracy": results["overall_accuracy"],
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./IndicBERTv3_POS",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=15,  # Increased number of epochs

    weight_decay=0.01,

    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none",
    fp16=False,  # Disabled mixed precision training due to BFloat16 compatibility error
    warmup_steps=899,  # Replaced warmup_ratio with calculated warmup_steps (0.1 * (4791 / 8) * 15 = 898.5, rounded to 899)
    # gradient_accumulation_steps=2 # Uncomment and adjust if you need to simulate larger batch sizes
)

In [ ]:
model = IndicBERTv3ForPOSTagging(
    MODEL_NAME,
    num_labels=len(label2id)
)

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Manually untied lm_head and embed_tokens weights via cloning.


In [ ]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    processing_class=tokenizer,

    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.247697,1.128999,0.682318,0.576466,0.507330,0.539693
2,0.812317,0.752740,0.775238,0.689700,0.666251,0.677772
3,0.700176,0.677506,0.798097,0.713945,0.700250,0.707031
4,0.557118,0.651212,0.805634,0.728304,0.710699,0.719394
5,0.450648,0.666331,0.806376,0.728328,0.714130,0.721159


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: RB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: V_VM seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: RD_PUNC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.247697,1.128999,0.682318,0.576466,0.507330,0.539693
2,0.812317,0.752740,0.775238,0.689700,0.666251,0.677772
3,0.700176,0.677506,0.798097,0.713945,0.700250,0.707031
4,0.557118,0.651212,0.805634,0.728304,0.710699,0.719394
5,0.450648,0.666331,0.806376,0.728328,0.714130,0.721159
6,0.429141,0.684762,0.808229,0.730457,0.715533,0.722918
7,0.362942,0.721628,0.806623,0.726023,0.716625,0.721293
8,0.341916,0.755390,0.801186,0.718647,0.712258,0.715438
9,0.280259,0.791454,0.802175,0.719531,0.708983,0.714218
10,0.271305,0.815024,0.800816,0.716859,0.707580,0.712189


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: RB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: V_VM seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: RD_PUNC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

TrainOutput(global_step=8985, training_loss=0.6796863110780583, metrics={'train_runtime': 2659.6479, 'train_samples_per_second': 27.02, 'train_steps_per_second': 3.378, 'total_flos': 0.0, 'train_loss': 0.6796863110780583, 'epoch': 15.0})

In [ ]:
trainer.evaluate(test_dataset)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NST seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: QT_QTC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:1

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.247792,0.720974,15,0.799292,0.714031,0.704782,0.709376


{'eval_loss': 0.7209739685058594,
 'eval_accuracy': 0.7992917667889212,
 'eval_precision': 0.7140311082307194,
 'eval_recall': 0.7047817047817048,
 'eval_f1': 0.7093762575452717}

### Per-Tag Performance Metrics

Now, let's analyze the performance of the model for each individual Part-of-Speech (POS) tag. This will give us a more detailed understanding of which tags the model is performing well on and which ones might need further improvement.

In [42]:
predictions_output = trainer.predict(test_dataset)
predictions = predictions_output.predictions
labels = predictions_output.label_ids

# Convert predictions to predicted tag IDs
predictions = np.argmax(predictions, axis=2)

true_predictions = []
true_labels = []

# Align predictions and labels, filtering out ignored tokens (-100)
for prediction, label in zip(predictions, labels):
    pred_tags = []
    gold_tags = []
    for pred, lab in zip(prediction, label):
        if lab != -100:
            pred_tags.append(id2label[pred])
            gold_tags.append(id2label[lab])
    true_predictions.append(pred_tags)
    true_labels.append(gold_tags)

# Compute detailed per-tag metrics
per_tag_metrics = seqeval.compute(predictions=true_predictions, references=true_labels)

print("Per-Tag Performance Metrics:")
for tag, metrics in per_tag_metrics.items():
    # Only print for actual tags, not overall metrics
    if isinstance(metrics, dict) and 'precision' in metrics:
        print(f"\nTag: {tag}")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall:    {metrics['recall']:.4f}")
        print(f"  F1-Score:  {metrics['f1']:.4f}")
        # Corrected from 'support' to 'number'
        print(f"  Support:   {metrics['number']}")

# Display overall metrics again for context if needed
print("\nOverall Metrics from seqeval.compute:")
print(f"  Overall Accuracy: {per_tag_metrics['overall_accuracy']:.4f}")
print(f"  Overall Precision: {per_tag_metrics['overall_precision']:.4f}")
print(f"  Overall Recall: {per_tag_metrics['overall_recall']:.4f}")
print(f"  Overall F1: {per_tag_metrics['overall_f1']:.4f}")

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: N_NST seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: QT_QTC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:1

Per-Tag Performance Metrics:

Tag: B
  Precision: 0.5496
  Recall:    0.5106
  F1-Score:  0.5294
  Support:   141

Tag: C_CCD
  Precision: 0.8700
  Recall:    0.9255
  F1-Score:  0.8969
  Support:   188

Tag: C_CCS
  Precision: 0.7347
  Recall:    0.5902
  F1-Score:  0.6545
  Support:   61

Tag: D_ECH
  Precision: 0.8125
  Recall:    0.2653
  F1-Score:  0.4000
  Support:   49

Tag: D_PUNC
  Precision: 0.9864
  Recall:    0.9895
  F1-Score:  0.9880
  Support:   954

Tag: D_RDF
  Precision: 0.9211
  Recall:    0.9459
  F1-Score:  0.9333
  Support:   74

Tag: D_SYM
  Precision: 1.0000
  Recall:    0.3333
  F1-Score:  0.5000
  Support:   12

Tag: D_UNK
  Precision: 0.3158
  Recall:    0.2222
  F1-Score:  0.2609
  Support:   27

Tag: J
  Precision: 0.5930
  Recall:    0.5000
  F1-Score:  0.5425
  Support:   472

Tag: M_DMD
  Precision: 0.9307
  Recall:    0.8995
  F1-Score:  0.9148
  Support:   209

Tag: M_DMI
  Precision: 0.1000
  Recall:    0.1250
  F1-Score:  0.1111
  Support:   8

Tag: 

In [ ]:
  trainer.save_model("./IndicBERTv3_Bodo_POS")
tokenizer.save_pretrained("./IndicBERTv3_Bodo_POS")

('./IndicBERTv3_Bodo_POS/tokenizer_config.json',
 './IndicBERTv3_Bodo_POS/tokenizer.json')

### Dataset Statistics

- **Total number of sentences:** After cleaning and deduplication, the dataset contains `5989` sentences.
- **Distribution of Part-of-Speech (POS) tags:** The following table shows the frequency of each POS tag in the dataset:

In [ ]:
print(f"Total number of sentences in the dataset: {df.shape[0]}")
print("\nDistribution of each tag in the dataset:")
display(tag_distribution)

Here are some additional details about the dataset:

- **Total number of sentences after cleaning:** This refers to the total number of unique entries in your dataset after removing duplicates.
- **Total number of unique Part-of-Speech (POS) tags:** This is the count of all distinct grammatical tags identified across the dataset.
- **Top 10 most frequent POS tags:** These are the most commonly occurring grammatical categories in your text data.
- **Bottom 10 least frequent POS tags:** These are the least commonly occurring grammatical categories, which might indicate rare linguistic phenomena or potential areas for further data collection if their frequency is unexpectedly low.

In [ ]:
print(f"Total number of sentences after cleaning: {df.shape[0]}")
print(f"Total number of unique POS tags: {len(tag_distribution)}")

print("\nTop 10 most frequent POS tags:")
display(tag_distribution.head(10))

print("\nBottom 10 least frequent POS tags:")
display(tag_distribution.tail(10))